In [ ]:
import pandas as pd
from statsmodels.tsa.stattools import adfuller
from prophet import Prophet
from pmdarima import auto_arima  
import plotly.graph_objects as go
from sklearn.metrics import r2_score
import plotly.graph_objects as go
from pathlib import Path

In [ ]:
df_clustering = pd.read_csv(Path('../model/SUPERSTORE_CLUSTERING.csv'))
df_clustering['ORDER_DATE'] = pd.to_datetime(df_clustering['ORDER_DATE'])
df_temporal = df_clustering.groupby([df_clustering['ORDER_DATE'].dt.to_period('M'), 'CLUSTER'])['PROFIT'].sum().reset_index()
df_temporal['ORDER_DATE'] = df_temporal['ORDER_DATE'].dt.to_timestamp() 
df_temporal = df_temporal.sort_values('ORDER_DATE').reset_index(drop=True)

In [ ]:
df_temporal_cluster_0 = df_temporal[df_temporal['CLUSTER'] == 0].copy()
df_temporal_cluster_1 = df_temporal[df_temporal['CLUSTER'] == 1].copy()
df_temporal_cluster_2 = df_temporal[df_temporal['CLUSTER'] == 2].copy()

df_temporal_dict = {0: df_temporal_cluster_0, 1: df_temporal_cluster_1, 2: df_temporal_cluster_2}

for i, data in df_temporal_dict.items():
    fig = go.Figure()
    fig.add_trace(
        go.Scatter(
            x=data['ORDER_DATE'],
            y=data['PROFIT'],
            mode='lines+markers',
            name=f'Cluster {i}'
        )
    )
    fig.update_layout(
        title=f"Profit - Cluster {i}",
        xaxis_title="Order Date",
        yaxis_title="Profit",
    )
    fig.show()

In [ ]:
for i in df_temporal_dict:
    df_temporal_dict[i] = df_temporal_dict[i][df_temporal_dict[i]['ORDER_DATE'] >= '2020-07-01']

In [ ]:
def test_stationarity(timeseries, diff_order=0):
    result = adfuller(timeseries)
    print(f'  Diferenciação: {diff_order}x')
    print(f'  ADF Statistic: {result[0]:.4f}')
    print(f'  p-value: {result[1]:.4f}')
    if result[1] < 0.05:
        print(f"  Estacionária (d={diff_order})\n")
    else:
        print(f"  Não estacionária, diferenciando...\n")
        test_stationarity(timeseries.diff().dropna(), diff_order + 1)


for i, data in df_temporal_dict.items():
    print(f"Cluster {i}:")
    test_stationarity(data['PROFIT'])

In [ ]:
arima_results = {}

for i, data in df_temporal_dict.items():
    ts = data.set_index('ORDER_DATE')['PROFIT']
    ts = ts.asfreq('MS', fill_value=0)

    train = ts[:int(0.8 * len(ts))]
    test = ts[int(0.8 * len(ts)):]

    print(f"--- Cluster {i} ---")
    auto_model = auto_arima(train, seasonal=False, trace=True, verbose=0)

    forecast = auto_model.predict(n_periods=len(test))
    forecast_series = pd.Series(forecast, index=test.index)

    arima_results[i] = {'train': train, 'test': test, 'forecast': forecast_series, 'model': auto_model}
    print('\n')

In [ ]:
forecast_months = 3

for i, data in df_temporal_dict.items():
    ts = data.copy()

    model = Prophet()
    model.fit(ts.rename(columns={'ORDER_DATE': 'ds', 'PROFIT': 'y'}))
    future = model.make_future_dataframe(periods=forecast_months, freq='MS')
    forecast = model.predict(future)

    forecast_in_sample = forecast[forecast['ds'].isin(ts['ORDER_DATE'])]
    y_true = ts.set_index('ORDER_DATE').loc[forecast_in_sample['ds']]['PROFIT'].values
    y_pred = forecast_in_sample['yhat'].values

    r2 = r2_score(y_true, y_pred)

    fig = go.Figure()
    fig.add_trace(go.Scatter(x=ts['ORDER_DATE'], y=ts['PROFIT'], mode='lines', name='Observado', line=dict(color='blue')))
    fig.add_trace(go.Scatter(x=forecast['ds'], y=forecast['yhat'], mode='lines', name='Previsão', line=dict(color='red', dash='dash')))
    fig.add_trace(go.Scatter(x=forecast['ds'], y=forecast['yhat_upper'], mode='lines', name='Upper', line=dict(width=0), showlegend=False))
    fig.add_trace(go.Scatter(x=forecast['ds'], y=forecast['yhat_lower'], mode='lines', name='Intervalo de Confiança', fill='tonexty', fillcolor='rgba(255,0,0,0.15)', line=dict(width=0)))
    
    mae = (y_true - y_pred).mean()
    mse = ((y_true - y_pred) ** 2).mean()

    fig.update_layout(
        title=f"Prophet Forecast - Cluster {i} ({forecast_months} meses) | R² = {r2:.3f} | MAE = {mae:.3f} | MSE = {mse:.3f}",
        xaxis_title="Order Date",
        yaxis_title="Profit",
        height=500
    )
    fig.show()